## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP

# Period 1 (baseline)
period1_start_date = '2023-10-01'
period1_end_date = '2023-12-31'

# Period 2 (comparison)
period2_start_date = '2024-01-01'
period2_end_date = '2024-03-31'

# Product filters
category_filter = 'Laundry'
flow_granularity = 'jp_brand_alter_lang_name'  # or jp_sub_brand_alter_lang_name, jp_prod_name

print(f"✓ Parameters set")
print(f"  Period 1: {period1_start_date} to {period1_end_date}")
print(f"  Period 2: {period2_start_date} to {period2_end_date}")
print(f"  Category: {category_filter}")
print(f"  Flow Level: {flow_granularity}")

## 3. Build and Execute Query

In [ ]:
# Build shopper flow query
query = f"""
WITH base AS (
    SELECT
        idpos.shopper_key AS shopper_id,
        {flow_granularity} AS product_name,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{period1_start_date}' AND '{period2_end_date}'
        AND shopper.member_ind = 'Y'
        AND {flow_granularity} IS NOT NULL
),
period1 AS (
    SELECT
        shopper_id,
        product_name,
        SUM(value) AS total_value,
        SUM(unit) AS total_unit
    FROM base
    WHERE purchase_date BETWEEN '{period1_start_date}' AND '{period1_end_date}'
    GROUP BY shopper_id, product_name
),
period1_top AS (
    SELECT
        shopper_id,
        product_name AS period1_product,
        ROW_NUMBER() OVER (PARTITION BY shopper_id ORDER BY total_value DESC) AS rn
    FROM period1
    QUALIFY rn = 1
),
period2 AS (
    SELECT
        shopper_id,
        product_name,
        SUM(value) AS total_value,
        SUM(unit) AS total_unit
    FROM base
    WHERE purchase_date BETWEEN '{period2_start_date}' AND '{period2_end_date}'
    GROUP BY shopper_id, product_name
),
period2_top AS (
    SELECT
        shopper_id,
        product_name AS period2_product,
        ROW_NUMBER() OVER (PARTITION BY shopper_id ORDER BY total_value DESC) AS rn
    FROM period2
    QUALIFY rn = 1
)
SELECT
    p1.period1_product,
    p2.period2_product,
    COUNT(DISTINCT p1.shopper_id) AS shopper_count,
    SUM(p2_detail.total_value) AS total_value,
    SUM(p2_detail.total_unit) AS total_unit
FROM period1_top p1
INNER JOIN period2_top p2 ON p1.shopper_id = p2.shopper_id
LEFT JOIN period2 p2_detail ON p2.shopper_id = p2_detail.shopper_id AND p2.period2_product = p2_detail.product_name
GROUP BY p1.period1_product, p2.period2_product
ORDER BY shopper_count DESC
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
df['shopper_count'] = df['shopper_count'].astype('Int64')
df['total_value'] = pd.to_numeric(df['total_value'], errors='coerce')
df['total_unit'] = pd.to_numeric(df['total_unit'], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} flow paths")
print(f"  Total shoppers tracked: {df['shopper_count'].sum():,}")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Calculate key metrics
total_shoppers = df['shopper_count'].sum()
loyalty_shoppers = df[df['period1_product'] == df['period2_product']]['shopper_count'].sum()
switching_shoppers = total_shoppers - loyalty_shoppers
loyalty_rate = (loyalty_shoppers / total_shoppers * 100) if total_shoppers > 0 else 0
switching_rate = (switching_shoppers / total_shoppers * 100) if total_shoppers > 0 else 0

# Top switching patterns
switching_df = df[df['period1_product'] != df['period2_product']].copy()
switching_df = switching_df.sort_values('shopper_count', ascending=False)

print("=" * 60)
print("SHOPPER FLOW ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTotal Shoppers Tracked: {total_shoppers:,}")
print(f"Loyal Shoppers (same product): {loyalty_shoppers:,} ({loyalty_rate:.1f}%)")
print(f"Switching Shoppers: {switching_shoppers:,} ({switching_rate:.1f}%)")
print(f"\nTop 5 Switching Patterns (Period 1 → Period 2):")
for idx, row in switching_df.head(5).iterrows():
    print(f"  {row['period1_product']:25s} → {row['period2_product']:25s}: {row['shopper_count']:5,} shoppers")

# Show loyalty patterns
loyalty_df = df[df['period1_product'] == df['period2_product']].sort_values('shopper_count', ascending=False)
print(f"\nTop 5 Loyal Brands:")
for idx, row in loyalty_df.head(5).iterrows():
    print(f"  {row['period1_product']:30s}: {row['shopper_count']:5,} shoppers")

## 5. Visualizations

In [ ]:
# Sankey diagram for shopper flow (top flows only)
top_n_flows = 30  # Show top 30 flows
sankey_df = df.nlargest(top_n_flows, 'shopper_count').copy()

# Create unique node lists
period1_nodes = sorted(sankey_df['period1_product'].unique())
period2_nodes = sorted(sankey_df['period2_product'].unique())
all_nodes = period1_nodes + period2_nodes

# Create node index mappings
node_dict = {node: idx for idx, node in enumerate(all_nodes)}

# Prepare link data
sources = [node_dict[p1] for p1 in sankey_df['period1_product']]
targets = [node_dict[p2] + len(period1_nodes) for p2 in sankey_df['period2_product']]
values = sankey_df['shopper_count'].tolist()

# Color loyal vs switching flows differently
link_colors = []
for idx, row in sankey_df.iterrows():
    if row['period1_product'] == row['period2_product']:
        link_colors.append('rgba(50, 205, 50, 0.4)')  # Green for loyalty
    else:
        link_colors.append('rgba(255, 140, 0, 0.3)')  # Orange for switching

# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=[f"{node}<br>(P1)" if node in period1_nodes else f"{node}<br>(P2)" for node in all_nodes],
        color=['lightblue'] * len(period1_nodes) + ['lightcoral'] * len(period2_nodes)
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors
    )
)])

fig.update_layout(
    title=f"Shopper Flow: {category_filter} Category<br>Period 1 ({period1_start_date} to {period1_end_date}) → Period 2 ({period2_start_date} to {period2_end_date})",
    font_size=10,
    height=800
)

fig.show()

In [ ]:
# Loyalty vs Switching comparison chart
comparison_data = pd.DataFrame({
    'Behavior': ['Loyal', 'Switching'],
    'Shoppers': [loyalty_shoppers, switching_shoppers],
    'Percentage': [loyalty_rate, switching_rate]
})

fig = px.pie(
    comparison_data,
    values='Shoppers',
    names='Behavior',
    title='Loyalty vs Switching Behavior',
    color='Behavior',
    color_discrete_map={'Loyal': '#32CD32', 'Switching': '#FF8C00'},
    hole=0.4
)

fig.update_traces(
    textposition='inside',
    texttemplate='%{label}<br>%{value:,}<br>(%{percent})',
    textfont_size=14
)

fig.update_layout(height=500)
fig.show()

## 6. Data Tables

In [ ]:
# Display switching patterns
print("Top 20 Switching Patterns:")
switching_df.head(20)

In [ ]:
# Display loyalty patterns
print("Top 10 Loyal Brands:")
loyalty_df.head(10)

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"shopper_flow_{category_filter}_{period1_start_date}_to_{period2_end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': [
            'Total Shoppers',
            'Loyal Shoppers',
            'Loyalty Rate %',
            'Switching Shoppers',
            'Switching Rate %'
        ],
        'Value': [
            f"{total_shoppers:,}",
            f"{loyalty_shoppers:,}",
            f"{loyalty_rate:.1f}%",
            f"{switching_shoppers:,}",
            f"{switching_rate:.1f}%"
        ]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # All flows
    df.to_excel(writer, sheet_name='All_Flows', index=False)
    
    # Switching patterns
    switching_df.to_excel(writer, sheet_name='Switching_Patterns', index=False)
    
    # Loyalty patterns
    loyalty_df.to_excel(writer, sheet_name='Loyalty_Patterns', index=False)

print(f"✓ Data exported to: {export_file}")